# Closing the Gap: Advanced FE + Multi-Loss Ensemble 🏁

This notebook combines every technique that proved effective during this competition into one pipeline:
- **Multi-scale target encoding** with Gaussian kernel smoothing
- **Frequency encoding** and **interaction features**
- **Three diverse models**: LightGBM (LogLoss), LightGBM (Focal Loss), XGBoost
- **OOF rank blending** with weight optimization

The goal: squeeze every last drop of AUC from this dataset.

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
from functools import partial
import lightgbm as lgb
import xgboost as xgb

sns.set_style('whitegrid')

# Auto-detect data path
for path in ['/kaggle/input/playground-series-s6e9',
             '/kaggle/input/competitions/playground-series-s6e9', '.']:
    if os.path.exists(os.path.join(path, 'train.csv')):
        DATA_DIR = path
        break

train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
print(f'Train: {train.shape}, Test: {test.shape}')

## Feature Engineering

In [ ]:
# --- Encode categoricals ---
for df in [train, test]:
    df['Home_Charging_Possible'] = df['Home_Charging_Possible'].map({'Yes': 1, 'No': 0})
    df['Subsidy_Available'] = df['Subsidy_Available'].map({'Yes': 1, 'No': 0})
    df['Range_Anxiety_Level'] = df['Range_Anxiety_Level'].map({'Low': 0, 'Medium': 1, 'High': 3})
    df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1, 'Other': 2})
    df['City_Type'] = df['City_Type'].map({'Rural': 0, 'Suburban': 1, 'Urban': 2})
    df['Current_Car_Type'] = df['Current_Car_Type'].map({'Hatchback': 0, 'SUV': 1, 'Sedan': 2, 'Truck': 3})

train['Will_Buy_EV'] = train['Will_Buy_EV'].map({'Yes': 1, 'No': 0})

# --- Interaction features ---
for df in [train, test]:
    df['homecharge'] = df['Charging_Stations_Near_Home'] + 5 * df['Home_Charging_Possible']
    df['subsidy_x_concern'] = df['Subsidy_Available'] * df['Environmental_Concern_Level']
    df['subsidy_x_income'] = df['Subsidy_Available'] * df['Annual_Income_USD']
    df['concern_x_anxiety'] = df['Environmental_Concern_Level'] * df['Range_Anxiety_Level']
    df['charging_total'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
    df['subsidy_x_home_charging'] = df['Subsidy_Available'] * df['Home_Charging_Possible']
    df['subsidy_x_anxiety'] = df['Subsidy_Available'] * df['Range_Anxiety_Level']
    df['income_per_car'] = df['Annual_Income_USD'] / (df['Number_of_Cars_Owned'] + 1)
    df['concern_x_income'] = df['Environmental_Concern_Level'] * df['Annual_Income_USD']
    df['anxiety_x_charging'] = df['Range_Anxiety_Level'] * df['charging_total']

y = train['Will_Buy_EV'].values
test_ids = test['id'].values

# Column groups for per-fold encoding
te_cols = ['Age', 'Gender', 'Number_of_Cars_Owned', 'City_Type',
           'Current_Car_Type', 'Range_Anxiety_Level']
tegauss_cols = ['Annual_Income_USD', 'Daily_Commute_km']
drop_cols = ['Age', 'Gender', 'Number_of_Cars_Owned', 'City_Type',
             'Current_Car_Type', 'Charging_Stations_Near_Home']

base_feature_cols = [c for c in train.columns if c not in ['id', 'Will_Buy_EV']]
X_base = train[base_feature_cols].copy()
X_test_base = test[base_feature_cols].copy()

print(f'Base features: {len(base_feature_cols)}')
print(f'Target: {np.mean(y):.4f} positive rate')

In [ ]:
def smoothed_target_encoding_gaussian(X_train_col, y_train, kernel_param):
    """Gaussian kernel-smoothed target encoding."""
    grouped = pd.Series(y_train).groupby(X_train_col).agg(['mean', 'count']).reset_index()
    cats = grouped[X_train_col.name].values.astype(float)
    means = grouped['mean'].values.astype(float)
    counts = grouped['count'].values.astype(float)
    diff = cats[:, None] - cats[None, :]
    K = np.exp(-kernel_param * diff * diff)
    numerator = np.sum(K * (means * counts)[None, :], axis=1)
    denominator = np.sum(K * counts[None, :], axis=1)
    denominator = np.maximum(denominator, 1e-12)
    smoothed = numerator / denominator
    return dict(zip(cats, smoothed))

def apply_fold_encodings(X_tr, y_tr, X_va, X_te):
    """Apply all encodings inside a CV fold. Fit on train, map to val/test."""
    X_train_enc = X_tr.copy()
    X_val_enc = X_va.copy()
    X_test_enc = X_te.copy()
    
    global_mean = y_tr.mean()
    
    # --- Standard smoothed target encoding ---
    for alpha_te in [0.5, 5.0]:  # Multi-scale: tight and loose smoothing
        suffix = f'_a{alpha_te}'
        for col in te_cols:
            counts = pd.Series(y_tr).groupby(X_tr[col]).size()
            means = pd.Series(y_tr).groupby(X_tr[col]).mean()
            smoothed = (counts * means + alpha_te * global_mean) / (counts + alpha_te)
            X_train_enc[f'te_{col}{suffix}'] = X_tr[col].map(smoothed).fillna(global_mean)
            X_val_enc[f'te_{col}{suffix}'] = X_va[col].map(smoothed).fillna(global_mean)
            X_test_enc[f'te_{col}{suffix}'] = X_te[col].map(smoothed).fillna(global_mean)
    
    # --- Frequency encoding ---
    for col in te_cols:
        freq_map = X_tr[col].value_counts().to_dict()
        X_train_enc[f'freq_{col}'] = X_tr[col].map(freq_map).fillna(0)
        X_val_enc[f'freq_{col}'] = X_va[col].map(freq_map).fillna(0)
        X_test_enc[f'freq_{col}'] = X_te[col].map(freq_map).fillna(0)
    
    # --- Gaussian kernel target encoding (multi-scale) ---
    for col in tegauss_cols:
        for kp in [0.02, 0.001]:  # Fine and coarse kernels
            gauss_map = smoothed_target_encoding_gaussian(X_tr[col], y_tr, kernel_param=kp)
            suffix = f'_k{kp}'
            X_train_enc[f'te_gauss_{col}{suffix}'] = X_tr[col].map(gauss_map).fillna(global_mean)
            X_val_enc[f'te_gauss_{col}{suffix}'] = X_va[col].map(gauss_map).fillna(global_mean)
            X_test_enc[f'te_gauss_{col}{suffix}'] = X_te[col].map(gauss_map).fillna(global_mean)
    
    # --- Drop raw categoricals (replaced by TE/freq) ---
    X_train_enc = X_train_enc.drop(columns=drop_cols, errors='ignore')
    X_val_enc = X_val_enc.drop(columns=drop_cols, errors='ignore')
    X_test_enc = X_test_enc.drop(columns=drop_cols, errors='ignore')
    
    return X_train_enc, X_val_enc, X_test_enc

# --- Focal Loss ---
def _focal_loss(preds, train_data, gamma, alpha):
    labels = train_data.get_label()
    p = 1.0 / (1.0 + np.exp(-preds))
    p = np.clip(p, 1e-7, 1 - 1e-7)
    pt = np.where(labels == 1, p, 1 - p)
    focal_weight = (1 - pt) ** gamma
    alpha_factor = np.where(labels == 1, alpha, 1 - alpha) * 2
    weight = alpha_factor * focal_weight
    grad = weight * (p - labels)
    hess = weight * p * (1 - p)
    hess = np.clip(hess, 1e-7, None)
    return grad, hess

focal_loss = partial(_focal_loss, gamma=1.2, alpha=0.65)

def pct_rank(v):
    return (rankdata(v) - 0.5) / len(v)

print('Encoding functions and focal loss ready.')

## Model Configurations

Three diverse models with carefully tuned parameters.

In [ ]:
# Model 1: LightGBM with LogLoss
lgb_logloss_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'num_leaves': 63,
    'learning_rate': 0.02,
    'feature_fraction': 0.3,
    'bagging_fraction': 0.9,
    'bagging_freq': 1,
    'reg_alpha': 0.05,
    'reg_lambda': 10.0,
    'max_bin': 1023,
    'min_data_in_leaf': 10,
    'linear_tree': True,
    'feature_pre_filter': False,
    'n_jobs': -1,
}

# Model 2: LightGBM with Focal Loss
lgb_focal_params = {
    'objective': focal_loss,
    'metric': 'auc',
    'verbose': -1,
    'num_leaves': 63,
    'learning_rate': 0.02,
    'feature_fraction': 0.3,
    'bagging_fraction': 0.9,
    'bagging_freq': 1,
    'reg_alpha': 0.05,
    'reg_lambda': 10.0,
    'max_bin': 1023,
    'min_data_in_leaf': 10,
    'linear_tree': True,
    'feature_pre_filter': False,
    'n_jobs': -1,
}

# Model 3: XGBoost
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'verbosity': 0,
    'max_depth': 7,
    'learning_rate': 0.02,
    'n_estimators': 5000,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'min_child_weight': 10,
    'reg_alpha': 0.05,
    'reg_lambda': 10.0,
    'early_stopping_rounds': 150,
    'random_state': 42,
    'n_jobs': -1,
}

print('3 model configurations defined.')

## 5-Fold Cross-Validation: All 3 Models

In [ ]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Storage
oof_lgb_ll = np.zeros(len(y))
oof_lgb_fl = np.zeros(len(y))
oof_xgb = np.zeros(len(y))

test_lgb_ll = np.zeros(len(X_test_base))
test_lgb_fl = np.zeros(len(X_test_base))
test_xgb = np.zeros(len(X_test_base))

scores = {'lgb_ll': [], 'lgb_fl': [], 'xgb': []}

for fold, (train_idx, val_idx) in enumerate(skf.split(X_base, y), 1):
    print(f'\n=== Fold {fold} ===')
    
    X_tr = X_base.iloc[train_idx].reset_index(drop=True)
    X_va = X_base.iloc[val_idx].reset_index(drop=True)
    y_tr, y_va = y[train_idx], y[val_idx]
    
    # Apply fold encodings
    X_tr_enc, X_va_enc, X_te_enc = apply_fold_encodings(X_tr, y_tr, X_va, X_test_base)
    
    # --- Model 1: LGB LogLoss ---
    dtrain1 = lgb.Dataset(X_tr_enc, label=y_tr)
    dval1 = lgb.Dataset(X_va_enc, label=y_va, reference=dtrain1)
    
    m1 = lgb.train(lgb_logloss_params, dtrain1,
                   num_boost_round=5000, valid_sets=[dval1],
                   callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    
    oof_lgb_ll[val_idx] = m1.predict(X_va_enc)
    test_lgb_ll += m1.predict(X_te_enc) / N_SPLITS
    s1 = roc_auc_score(y_va, oof_lgb_ll[val_idx])
    scores['lgb_ll'].append(s1)
    
    # --- Model 2: LGB Focal ---
    dtrain2 = lgb.Dataset(X_tr_enc, label=y_tr)
    dval2 = lgb.Dataset(X_va_enc, label=y_va, reference=dtrain2)
    
    m2 = lgb.train(lgb_focal_params, dtrain2,
                   num_boost_round=5000, valid_sets=[dval2],
                   callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    
    raw_va = m2.predict(X_va_enc)
    oof_lgb_fl[val_idx] = 1.0 / (1.0 + np.exp(-raw_va))
    raw_te = m2.predict(X_te_enc)
    test_lgb_fl += (1.0 / (1.0 + np.exp(-raw_te))) / N_SPLITS
    s2 = roc_auc_score(y_va, oof_lgb_fl[val_idx])
    scores['lgb_fl'].append(s2)
    
    # --- Model 3: XGBoost ---
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr_enc, y_tr,
                  eval_set=[(X_va_enc, y_va)], verbose=False)
    
    oof_xgb[val_idx] = model_xgb.predict_proba(X_va_enc)[:, 1]
    test_xgb += model_xgb.predict_proba(X_te_enc)[:, 1] / N_SPLITS
    s3 = roc_auc_score(y_va, oof_xgb[val_idx])
    scores['xgb'].append(s3)
    
    print(f'  LGB-LL: {s1:.6f}  |  LGB-FL: {s2:.6f}  |  XGB: {s3:.6f}')

print(f'\n{"="*60}')
print(f'  LGB LogLoss  OOF AUC: {roc_auc_score(y, oof_lgb_ll):.6f}  (mean: {np.mean(scores["lgb_ll"]):.6f})')
print(f'  LGB Focal    OOF AUC: {roc_auc_score(y, oof_lgb_fl):.6f}  (mean: {np.mean(scores["lgb_fl"]):.6f})')
print(f'  XGBoost      OOF AUC: {roc_auc_score(y, oof_xgb):.6f}  (mean: {np.mean(scores["xgb"]):.6f})')
print(f'{"="*60}')

## OOF Rank Blend: Finding Optimal Weights

With 3 models, we search a 2D weight space.

In [ ]:
# 3-model blend: search over 2D grid
best_auc = 0
best_w = (0.33, 0.33, 0.34)
results = []

for w1 in np.arange(0.0, 1.01, 0.05):
    for w2 in np.arange(0.0, 1.01 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0:
            continue
        blend = pct_rank(
            w1 * pct_rank(oof_lgb_ll) + 
            w2 * pct_rank(oof_lgb_fl) + 
            w3 * pct_rank(oof_xgb)
        )
        auc = roc_auc_score(y, blend)
        results.append((w1, w2, w3, auc))
        if auc > best_auc:
            best_auc = auc
            best_w = (w1, w2, w3)

# Also try pairwise blends
for w in np.linspace(0, 1, 101):
    # LGB-LL + LGB-FL
    blend = pct_rank(w * pct_rank(oof_lgb_ll) + (1-w) * pct_rank(oof_lgb_fl))
    auc = roc_auc_score(y, blend)
    if auc > best_auc:
        best_auc = auc
        best_w = (w, 1-w, 0.0)
    
    # LGB-LL + XGB
    blend = pct_rank(w * pct_rank(oof_lgb_ll) + (1-w) * pct_rank(oof_xgb))
    auc = roc_auc_score(y, blend)
    if auc > best_auc:
        best_auc = auc
        best_w = (w, 0.0, 1-w)

individual_aucs = {
    'LGB-LL': roc_auc_score(y, oof_lgb_ll),
    'LGB-FL': roc_auc_score(y, oof_lgb_fl),
    'XGB': roc_auc_score(y, oof_xgb),
}
best_single = max(individual_aucs.values())

print('='*60)
print('  BLEND RESULTS')
print('='*60)
for name, auc in individual_aucs.items():
    print(f'  {name:12s}: {auc:.6f}')
print(f'  {"Blend":12s}: {best_auc:.6f}  (+{best_auc - best_single:.6f} vs best single)')
print(f'  Weights: LGB-LL={best_w[0]:.2f}, LGB-FL={best_w[1]:.2f}, XGB={best_w[2]:.2f}')
print('='*60)

In [ ]:
# Apply optimal weights to test predictions
test_blend = pct_rank(
    best_w[0] * pct_rank(test_lgb_ll) + 
    best_w[1] * pct_rank(test_lgb_fl) + 
    best_w[2] * pct_rank(test_xgb)
)

submission = pd.DataFrame({'id': test_ids, 'Will_Buy_EV': test_blend})
submission.to_csv('submission.csv', index=False)

print(f'Submission shape: {submission.shape}')
print(f'Mean: {test_blend.mean():.4f}, Std: {test_blend.std():.4f}')
print(f'Min: {test_blend.min():.4f}, Max: {test_blend.max():.4f}')
submission.head()

## Summary

| Model | OOF AUC |
|---|---|
| LGB LogLoss | see output |
| LGB Focal Loss | see output |
| XGBoost | see output |
| **Rank-Avg Blend** | **see output** |

Key techniques:
1. **Multi-scale target encoding** (alpha=0.5 and alpha=5.0) captures both fine and coarse patterns
2. **Multi-scale Gaussian TE** (kernel=0.02 and 0.001) on continuous features
3. **Focal Loss** focuses gradient on hard-to-classify samples
4. **Model diversity** (LGB linear_tree vs XGB standard trees) creates orthogonality
5. **OOF rank blending** finds optimal model weights without leakage

Credits: Inspired by [@najiama](https://www.kaggle.com/najiama), [@sergeyqt2024](https://www.kaggle.com/sergeyqt2024), and [@cdeotte](https://www.kaggle.com/cdeotte).

**If you found this useful, please upvote! 👍**